# Business Forecasting — Lecture 4: The Forecaster’s Toolbox and ARIMA

This notebook accompanies Lecture 4. It moves from simple benchmark forecasts to ARIMA and seasonal ARIMA models.

**You only need the course data ZIP or the four CSV files.** The notebook does not download data from the internet.

Datasets used:
- `aus_production.csv` — quarterly Australian production data; we use **Bricks**.
- `apple_stock_daily.csv` — daily Apple stock prices; we use **Close** to illustrate stationarity and differencing.
- `global_economy.csv` — annual macroeconomic data; we use **Egyptian exports**.
- `us_employment.csv` — monthly US employment data; we use **Leisure and Hospitality**.

## 1. Setup and file upload

Run the next two cells first. In Google Colab, you can upload either:
1. the provided ZIP file, **or**
2. the four CSV files individually.

The code places all CSV files in a folder named `data`, so the rest of the notebook always reads from the same location.

In [ ]:
# Core packages used throughout the lecture.
from pathlib import Path
import zipfile
import shutil
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import kpss
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

In [ ]:
# Create one folder where all CSV files will live.
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# In Google Colab, open a file picker so students can upload a ZIP or CSV files.
try:
    from google.colab import files
    uploaded = files.upload()

    for filename, file_bytes in uploaded.items():
        source = Path(filename)
        source.write_bytes(file_bytes)

        # If a ZIP was uploaded, extract every CSV file into DATA_DIR.
        if source.suffix.lower() == ".zip":
            with zipfile.ZipFile(source, "r") as z:
                for member in z.namelist():
                    if member.lower().endswith(".csv"):
                        target = DATA_DIR / Path(member).name
                        target.write_bytes(z.read(member))

        # If an individual CSV was uploaded, copy it into DATA_DIR.
        elif source.suffix.lower() == ".csv":
            shutil.copy(source, DATA_DIR / source.name)

except ImportError:
    # This branch is used when the notebook is tested outside Google Colab.
    # Put the supplied CSV files inside a folder named 'data' before running.
    pass

print("CSV files available:")
for f in sorted(DATA_DIR.glob("*.csv")):
    print(" -", f.name)

## 2. Basic benchmark forecasting methods

Before ARIMA, we need simple methods that act as benchmarks.

- **Mean method:** every future forecast equals the mean of the training data.
- **Naive method:** every future forecast equals the last observed value.
- **Seasonal naive:** each forecast repeats the last observation from the same season.
- **Drift method:** extends the average historical change forward.

A complicated model should earn its complexity by performing better than these simple benchmarks.

In [ ]:
# Load quarterly Australian production data.
production = pd.read_csv(DATA_DIR / "aus_production.csv", parse_dates=["ds"])

# Keep only the date and Bricks columns and remove missing observations.
bricks = production[["ds", "Bricks"]].dropna().reset_index(drop=True)

# reset_index(drop=True) gives the filtered table a clean row index 0, 1, 2, ...
# This makes later row-based operations easier to read.

print(bricks.head())
print("Number of quarterly observations:", len(bricks))

In [ ]:
# Plot the historical series before forecasting it.
plt.plot(bricks["ds"], bricks["Bricks"])
plt.title("Australian quarterly brick production")
plt.xlabel("Quarter")
plt.ylabel("Bricks production")
plt.show()

### Chronological train/test split

For time series, we do **not** randomly shuffle observations. The model must learn from the past and be evaluated on later observations.

Here we use roughly the first 80% as training data and the last 20% as test data.

In [ ]:
# Find the row where the 80% training portion ends.
split = int(len(bricks) * 0.80)

# iloc selects rows by position.
# Everything before 'split' becomes training data.
train = bricks.iloc[:split].copy()

# Everything from 'split' onward becomes test data.
test = bricks.iloc[split:].copy()

# copy() creates independent working DataFrames.
# It prevents later edits from unintentionally modifying the original table.

print("Training period:", train["ds"].min(), "to", train["ds"].max())
print("Test period:", test["ds"].min(), "to", test["ds"].max())

### Create the four benchmark forecasts

Notice that each method uses **only the training data**. The test values remain hidden until evaluation.

In [ ]:
h = len(test)  # h is the forecast horizon: number of test periods.

# MEAN METHOD: repeat the training mean for all future periods.
mean_forecast = np.repeat(train["Bricks"].mean(), h)

# NAIVE METHOD: repeat the last observed training value.
naive_forecast = np.repeat(train["Bricks"].iloc[-1], h)

# SEASONAL NAIVE: quarterly data have season length 4.
# Take the final four training quarters and repeat Q1, Q2, Q3, Q4 as needed.
last_four = train["Bricks"].iloc[-4:].to_numpy()
seasonal_naive_forecast = np.array([last_four[i % 4] for i in range(h)])

# DRIFT METHOD: estimate the average change per quarter from first to last training value.
drift_per_period = (
    train["Bricks"].iloc[-1] - train["Bricks"].iloc[0]
) / (len(train) - 1)

# Extend that average change h periods into the future.
drift_forecast = train["Bricks"].iloc[-1] + drift_per_period * np.arange(1, h + 1)

In [ ]:
# Plot actual values and all four forecasts over the test period.
plt.plot(bricks["ds"], bricks["Bricks"], label="Actual")
plt.plot(test["ds"], mean_forecast, label="Mean")
plt.plot(test["ds"], naive_forecast, label="Naive")
plt.plot(test["ds"], seasonal_naive_forecast, label="Seasonal naive")
plt.plot(test["ds"], drift_forecast, label="Drift")
plt.axvline(test["ds"].iloc[0], linestyle="--", label="Test begins")
plt.title("Benchmark forecasts")
plt.xlabel("Quarter")
plt.ylabel("Bricks production")
plt.legend()
plt.show()

### Forecast accuracy

We will use:
- **MAE:** average absolute error.
- **RMSE:** square root of average squared error; large mistakes receive extra weight.
- **MAPE:** average absolute percentage error; easy to interpret, but problematic when actual values are zero or very close to zero.

Lower values indicate better accuracy **when models are compared on the same test set**.

In [ ]:
def accuracy_metrics(actual, forecast):
    """Return MAE, RMSE and MAPE for one set of forecasts."""
    actual = np.asarray(actual)
    forecast = np.asarray(forecast)

    error = actual - forecast
    mae = np.mean(np.abs(error))
    rmse = np.sqrt(np.mean(error ** 2))
    mape = np.mean(np.abs(error / actual)) * 100

    return {"MAE": mae, "RMSE": rmse, "MAPE": mape}

results = []
for name, forecast in {
    "Mean": mean_forecast,
    "Naive": naive_forecast,
    "Seasonal naive": seasonal_naive_forecast,
    "Drift": drift_forecast,
}.items():
    row = accuracy_metrics(test["Bricks"], forecast)
    row["Method"] = name
    results.append(row)

accuracy_table = pd.DataFrame(results)[["Method", "MAE", "RMSE", "MAPE"]]
accuracy_table.sort_values("RMSE")

## 3. Residuals versus forecast errors

These two terms are related but not identical.

- A **residual** is calculated on the training data: actual minus fitted value.
- A **forecast error** is calculated on unseen test/future data: actual minus forecast.

Residuals are mainly used for **diagnostics**. Forecast errors are mainly used for **out-of-sample accuracy**.

In [ ]:
# For a one-step naive fit, the fitted value at time t is the value at time t-1.
train["Naive_fitted"] = train["Bricks"].shift(1)

# Residual = actual - fitted.
train["Residual"] = train["Bricks"] - train["Naive_fitted"]

# The first residual is missing because there is no previous observation for the first row.
residuals = train["Residual"].dropna()

print("Mean residual:", residuals.mean())

In [ ]:
# A good residual series should look random around zero.
fig, axes = plt.subplots(2, 1, figsize=(10, 7))

axes[0].plot(train.loc[residuals.index, "ds"], residuals)
axes[0].axhline(0)
axes[0].set_title("Naive-method residuals")
axes[0].set_ylabel("Residual")

# The ACF checks whether residuals are correlated with their own past.
plot_acf(residuals, lags=20, zero=False, ax=axes[1], bartlett_confint=False)
axes[1].set_title("Residual ACF")
axes[1].set_xlabel("Lag")
axes[1].set_ylabel("ACF")

plt.tight_layout()
plt.show()

In [ ]:
# Ljung-Box tests whether a group of residual autocorrelations is jointly zero.
# H0: residuals are uncorrelated up to the chosen lag.
# A small p-value suggests useful pattern remains in the residuals.
ljung_naive = acorr_ljungbox(residuals, lags=[4, 8, 12], return_df=True)
ljung_naive

## 4. Stationarity — the starting point for ARIMA

A stationary time series has statistical behaviour that is reasonably stable over time: roughly stable level, variability, and dependence structure.

Trend and fixed seasonality usually indicate non-stationarity. ARIMA often handles this by **differencing** the data before modelling autocorrelation.

In [ ]:
# Load Apple daily stock prices and sort them in chronological order.
apple = pd.read_csv(DATA_DIR / "apple_stock_daily.csv", parse_dates=["Date"])
apple = apple.sort_values("Date").reset_index(drop=True)

# First difference = today's close minus yesterday's close.
apple["Change"] = apple["Close"].diff()

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].plot(apple["Date"], apple["Close"])
axes[0].set_title("Apple closing price")
axes[0].set_ylabel("Price")

axes[1].plot(apple["Date"], apple["Change"])
axes[1].axhline(0)
axes[1].set_title("First difference: daily change")
axes[1].set_ylabel("Change")
axes[1].set_xlabel("Date")

plt.tight_layout()
plt.show()

### Why differencing helps

The original stock-price level moves slowly and persistently. Consecutive values are therefore highly correlated.

The first difference changes the question from **“What is the price?”** to **“How much did the price change since yesterday?”** This often removes changing level and makes the series closer to stationary.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

plot_acf(
    apple["Close"], lags=30, zero=False,
    ax=axes[0], bartlett_confint=False
)
axes[0].set_title("ACF: stock-price level")

plot_acf(
    apple["Change"].dropna(), lags=30, zero=False,
    ax=axes[1], bartlett_confint=False
)
axes[1].set_title("ACF: first differences")

plt.tight_layout()
plt.show()

### KPSS stationarity test

For the KPSS test:
- **H0:** the series is stationary.
- A **small p-value** gives evidence against stationarity.

Statistical tests should support—not replace—visual inspection of the time plot and ACF.

In [ ]:
def run_kpss(series, name):
    stat, p_value, *_ = kpss(series.dropna(), nlags="auto")
    print(f"{name}: KPSS statistic = {stat:.3f}, p-value = {p_value:.3f}")

run_kpss(apple["Close"], "Original close")
run_kpss(apple["Change"], "First difference")

### Random walk connection

If first differences are essentially unpredictable white noise,

\[
y_t - y_{t-1} = arepsilon_t
\]

then

\[
y_t = y_{t-1} + arepsilon_t.
\]

This is a **random walk**. Its optimal point forecast is the most recent observation—which is exactly the **naive method**.

## 5. Seasonal differencing

For seasonal data, first differencing may not remove the repeating seasonal pattern.

A seasonal difference compares an observation with the observation from the **same season in the previous cycle**:

\[
y_t - y_{t-m}
\]

For monthly data with annual seasonality, `m = 12`. For quarterly data, `m = 4`.

In [ ]:
# Load monthly US employment and select Leisure and Hospitality from 1990 onward.
employment = pd.read_csv(DATA_DIR / "us_employment.csv", parse_dates=["ds"])

# query() filters rows using a readable condition.
# We need only one industry because ARIMA here is a univariate time-series model.
leisure = employment.query(
    "unique_id == 'Leisure and Hospitality' and ds >= '1990-01-01'"
).copy()

# copy() makes an independent table that we can safely modify.
leisure = leisure.sort_values("ds").reset_index(drop=True)
leisure["Millions"] = leisure["y"] / 1000

# Seasonal difference: compare each month with the same month one year earlier.
leisure["Seasonal_diff"] = leisure["Millions"].diff(12)

# First difference after seasonal differencing.
leisure["Double_diff"] = leisure["Seasonal_diff"].diff()

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
axes[0].plot(leisure["ds"], leisure["Millions"])
axes[0].set_title("Original monthly series")
axes[1].plot(leisure["ds"], leisure["Seasonal_diff"])
axes[1].set_title("Seasonal difference: lag 12")
axes[2].plot(leisure["ds"], leisure["Double_diff"])
axes[2].set_title("Seasonal + first difference")
axes[2].set_xlabel("Month")
plt.tight_layout()
plt.show()

**Important:** do as little differencing as necessary. Over-differencing can create artificial autocorrelation and make a model harder to interpret.

## 6. AR, MA and ARIMA models

### Autoregressive model — AR(p)
An AR model predicts the current value using **past values of the same series**.

For AR(1):

\[
y_t = c + \phi_1 y_{t-1} + arepsilon_t
\]

### Moving-average model — MA(q)
An MA model uses **past forecast errors (shocks)**:

\[
y_t = c + arepsilon_t + 	heta_1arepsilon_{t-1}+\cdots
\]

This is **not** the same as moving-average smoothing used to estimate a trend.

### ARIMA(p,d,q)

ARIMA combines:
- `p` = number of autoregressive lags,
- `d` = number of first differences,
- `q` = number of moving-average error lags.

The `I` in ARIMA means **Integrated**, which refers to reversing differencing when forecasts are returned to the original scale.

## 7. ACF and PACF for model identification

- **ACF** measures correlation between a series and its lagged values.
- **PACF** measures the direct relationship with lag `k` after removing the effects of shorter lags.

Useful textbook patterns:
- AR(p): PACF often cuts off after lag `p`, while ACF decays.
- MA(q): ACF often cuts off after lag `q`, while PACF decays.
- When both AR and MA terms are present, patterns can be less clean.

## 8. Non-seasonal ARIMA example — Egyptian exports

We use annual exports from `global_economy.csv`. `Exports` is measured as a percentage of GDP.

The table contains many countries, so we must separate Egypt before modelling.

In [ ]:
economy = pd.read_csv(DATA_DIR / "global_economy.csv")

# query() selects rows where the country name matches Egypt.
egypt = economy.query("unique_id == 'Egypt, Arab Rep.'")[["ds", "Exports"]]

# dropna() removes years where Exports is missing.
# copy() creates a separate working table.
# sort_values('ds') ensures chronological order.
egypt = (
    egypt.dropna()
         .copy()
         .sort_values("ds")
         .reset_index(drop=True)
)

plt.plot(egypt["ds"], egypt["Exports"], marker="o", markersize=3)
plt.title("Egyptian exports")
plt.xlabel("Year")
plt.ylabel("Exports (% of GDP)")
plt.show()

### Is differencing needed?

The plot does not show a strong deterministic trend. We use KPSS as an additional check. If the p-value is not small, we do not have evidence that first differencing is required, so `d = 0` is a reasonable starting point.

In [ ]:
stat, p_value, *_ = kpss(egypt["Exports"], nlags="auto")
print(f"KPSS statistic = {stat:.3f}")
print(f"p-value = {p_value:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

plot_acf(
    egypt["Exports"], lags=20, zero=False,
    ax=axes[0], bartlett_confint=False
)
axes[0].set_title("ACF")

plot_pacf(
    egypt["Exports"], lags=20, zero=False,
    ax=axes[1], method="ywm"
)
axes[1].set_title("PACF")

plt.tight_layout()
plt.show()

### Fit candidate ARIMA models

The book illustrates manual identification and automatic order selection. For this course notebook we use a transparent grid search with `statsmodels`.

Because KPSS suggests no first difference, we keep `d = 0` and compare reasonable `p` and `q` values. We use **AICc** to balance fit against unnecessary complexity.

In [ ]:
def aicc(fitted_model):
    """Small-sample corrected Akaike Information Criterion."""
    n = fitted_model.nobs
    k = len(fitted_model.params)
    return fitted_model.aic + (2 * k * (k + 1)) / (n - k - 1)

candidate_rows = []
candidate_fits = {}

for p in range(0, 5):
    for q in range(0, 4):
        if p == 0 and q == 0:
            continue

        try:
            model = ARIMA(
                egypt["Exports"],
                order=(p, 0, q),
                trend="c"
            )
            fit = model.fit()

            candidate_rows.append({
                "p": p,
                "d": 0,
                "q": q,
                "AIC": fit.aic,
                "AICc": aicc(fit),
                "BIC": fit.bic,
            })
            candidate_fits[(p, q)] = fit

        except Exception:
            # Some parameter combinations may fail to estimate.
            # We simply skip models that cannot be fitted reliably.
            pass

candidates = pd.DataFrame(candidate_rows).sort_values("AICc").reset_index(drop=True)
candidates.head(10)

`sort_values("AICc")` places the smallest AICc at the top. A lower AICc is preferred **among models fitted to the same data with the same differencing order**.

In [ ]:
best = candidates.iloc[0]
best_order = (int(best["p"]), int(best["d"]), int(best["q"]))
best_fit = candidate_fits[(best_order[0], best_order[2])]

print("Selected order:", best_order)
print("AICc:", round(best["AICc"], 3))
print(best_fit.summary())

## 9. Residual diagnostics for the selected ARIMA model

A model is not finished when it has a low AICc. We also want residuals that resemble **white noise**:
- centered around zero,
- no visible pattern,
- little remaining autocorrelation.

If residuals are predictable, the model has left information unused.

In [ ]:
arima_residuals = best_fit.resid

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

axes[0, 0].plot(egypt["ds"], arima_residuals)
axes[0, 0].axhline(0)
axes[0, 0].set_title("Residuals over time")

plot_acf(
    arima_residuals, lags=20, zero=False,
    ax=axes[0, 1], bartlett_confint=False
)
axes[0, 1].set_title("Residual ACF")

axes[1, 0].hist(arima_residuals, bins=14)
axes[1, 0].set_title("Residual distribution")

axes[1, 1].scatter(best_fit.fittedvalues, arima_residuals)
axes[1, 1].axhline(0)
axes[1, 1].set_title("Residuals vs fitted")

plt.tight_layout()
plt.show()

In [ ]:
# For Ljung-Box, model_df accounts for estimated AR and MA parameters.
model_df = best_order[0] + best_order[2]

ljung_arima = acorr_ljungbox(
    arima_residuals,
    lags=[10],
    model_df=model_df,
    return_df=True
)

ljung_arima

Interpretation: if the Ljung-Box p-value is comfortably above 0.05, we do not have strong evidence of remaining residual autocorrelation at the tested lags.

## 10. Forecast with prediction intervals

A point forecast is our central estimate. A prediction interval shows uncertainty around that estimate. The interval normally widens as the horizon increases because more future uncertainty accumulates.

In [ ]:
# Forecast the next 10 annual observations.
forecast_result = best_fit.get_forecast(steps=10)
forecast_frame = forecast_result.summary_frame(alpha=0.05)

future_years = np.arange(
    egypt["ds"].max() + 1,
    egypt["ds"].max() + 11
)

plt.plot(egypt["ds"], egypt["Exports"], label="Observed")
plt.plot(future_years, forecast_frame["mean"], label="Forecast")
plt.fill_between(
    future_years,
    forecast_frame["mean_ci_lower"],
    forecast_frame["mean_ci_upper"],
    alpha=0.2,
    label="95% prediction interval"
)
plt.title(f"Egyptian exports forecast — ARIMA{best_order}")
plt.xlabel("Year")
plt.ylabel("Exports (% of GDP)")
plt.legend()
plt.show()

## 11. Seasonal ARIMA

For seasonal data, ARIMA can include a second set of seasonal orders:

\[
ARIMA(p,d,q)(P,D,Q)_m
\]

- `p,d,q` describe non-seasonal dynamics.
- `P,D,Q` describe seasonal dynamics.
- `m` is the seasonal period, such as 12 for monthly data with yearly seasonality.

### Prepare the monthly employment series

We use the same Leisure and Hospitality data introduced earlier. We hold out the final 36 months so we can evaluate genuine multi-step forecasts.

In [ ]:
# Convert the date column into a monthly time index.
leisure_series = (
    leisure.set_index("ds")["Millions"]
           .asfreq("MS")
)

# Keep the final 36 months as a test set.
train_seasonal = leisure_series.iloc[:-36]
test_seasonal = leisure_series.iloc[-36:]

print("Training observations:", len(train_seasonal))
print("Test observations:", len(test_seasonal))

### Fit a seasonal ARIMA model

We fit `ARIMA(2,1,0)(1,1,1)[12]` as a worked classroom model.

- `d=1`: one ordinary difference.
- `D=1`: one seasonal difference.
- `m=12`: monthly annual seasonality.

`SARIMAX` is the `statsmodels` class that estimates seasonal ARIMA models.

In [ ]:
seasonal_model = SARIMAX(
    train_seasonal,
    order=(2, 1, 0),
    seasonal_order=(1, 1, 1, 12),
    trend="n",
    enforce_stationarity=False,
    enforce_invertibility=False,
)

seasonal_fit = seasonal_model.fit(disp=False)
print(seasonal_fit.summary())

### Diagnose seasonal-ARIMA residuals

State-space models can produce unusual initialization residuals at the very beginning. For a clean classroom diagnostic, we drop the first 24 residuals before examining the ACF and Ljung-Box test.

In [ ]:
seasonal_residuals = seasonal_fit.resid.dropna().iloc[24:]

fig, axes = plt.subplots(2, 1, figsize=(10, 7))
axes[0].plot(seasonal_residuals)
axes[0].axhline(0)
axes[0].set_title("Seasonal ARIMA residuals")

plot_acf(
    seasonal_residuals, lags=36, zero=False,
    ax=axes[1], bartlett_confint=False
)
axes[1].set_title("Residual ACF")

plt.tight_layout()
plt.show()

# Four fitted AR/MA parameters are accounted for in model_df.
acorr_ljungbox(
    seasonal_residuals,
    lags=[24],
    model_df=4,
    return_df=True
)

### Test-set forecast

This is a genuine evaluation: the model is fitted only to the training period, then forecasts 36 months it has not seen.

In [ ]:
seasonal_forecast = seasonal_fit.get_forecast(steps=len(test_seasonal))
seasonal_mean = seasonal_forecast.predicted_mean
seasonal_ci = seasonal_forecast.conf_int(alpha=0.05)

plt.plot(train_seasonal.index[-72:], train_seasonal.iloc[-72:], label="Training: last 6 years")
plt.plot(test_seasonal.index, test_seasonal, label="Test actual")
plt.plot(test_seasonal.index, seasonal_mean, label="Seasonal ARIMA forecast")
plt.fill_between(
    test_seasonal.index,
    seasonal_ci.iloc[:, 0],
    seasonal_ci.iloc[:, 1],
    alpha=0.2,
    label="95% interval"
)
plt.title("36-month seasonal ARIMA test-set forecast")
plt.xlabel("Month")
plt.ylabel("Employment (millions)")
plt.legend()
plt.show()

### Compare against seasonal naive

A seasonal ARIMA model should not be judged in isolation. We compare it with a simple seasonal-naive benchmark on exactly the same 36-month test period.

In [ ]:
# Repeat the final 12 months of training three times to cover the 36-month test set.
seasonal_naive_test = np.resize(train_seasonal.iloc[-12:].to_numpy(), len(test_seasonal))

comparison = pd.DataFrame([
    {
        "Method": "Seasonal naive",
        **accuracy_metrics(test_seasonal, seasonal_naive_test),
    },
    {
        "Method": "Seasonal ARIMA",
        **accuracy_metrics(test_seasonal, seasonal_mean),
    },
])

comparison

## 12. Final forecast using all available data

After choosing and checking a model, refit it using the full historical series. Then produce the actual future forecast.

In [ ]:
final_model = SARIMAX(
    leisure_series,
    order=(2, 1, 0),
    seasonal_order=(1, 1, 1, 12),
    trend="n",
    enforce_stationarity=False,
    enforce_invertibility=False,
)
final_fit = final_model.fit(disp=False)

future = final_fit.get_forecast(steps=36)
future_mean = future.predicted_mean
future_ci = future.conf_int(alpha=0.05)

plt.plot(leisure_series.index[-120:], leisure_series.iloc[-120:], label="Observed")
plt.plot(future_mean.index, future_mean, label="Forecast")
plt.fill_between(
    future_mean.index,
    future_ci.iloc[:, 0],
    future_ci.iloc[:, 1],
    alpha=0.2,
    label="95% interval"
)
plt.title("US Leisure and Hospitality employment — 3-year forecast")
plt.xlabel("Month")
plt.ylabel("Employment (millions)")
plt.legend()
plt.show()

# What to remember for later

### A. Open and prepare a CSV
```python
df = pd.read_csv("my_file.csv", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)
```

### B. Select one series from a larger table
```python
one_series = df.query("unique_id == 'Some name'").copy()
```
`query()` filters rows. `copy()` creates an independent working DataFrame.

### C. First and seasonal differences
```python
df["first_diff"] = df["y"].diff()
df["seasonal_diff"] = df["y"].diff(12)
```

### D. ACF and PACF
```python
plot_acf(series.dropna(), lags=24, zero=False)
plot_pacf(series.dropna(), lags=24, zero=False)
```

### E. Non-seasonal ARIMA
```python
fit = ARIMA(series, order=(p, d, q)).fit()
forecast = fit.get_forecast(steps=h)
```

### F. Seasonal ARIMA
```python
fit = SARIMAX(
    series,
    order=(p, d, q),
    seasonal_order=(P, D, Q, m)
).fit(disp=False)
```

### G. Residual check
```python
residuals = fit.resid.dropna()
plot_acf(residuals, lags=24, zero=False)
acorr_ljungbox(residuals, lags=[24], return_df=True)
```

### H. Accuracy must be measured on unseen data
```python
error = actual_test - forecast_test
```
Use the same test period when comparing models.

## Practice

1. Using `aus_production.csv`, choose another variable such as Beer or Cement. Create an 80/20 split and compare mean, naive, seasonal-naive and drift forecasts.
2. Examine the ACF of the training data. Does it suggest strong persistence or seasonality?
3. Fit one ARIMA model and compare its test RMSE with the best benchmark.
4. Check residuals. If the Ljung-Box p-value is small, explain what that tells you about the model.